# Notebook 01: Bronze Ingestion
## Purpose: Load raw NASA CMAPSS + Azure datasets into Bronze Delta tables
## Output Tables: bronze.nasa_sensor_raw, bronze.nasa_rul_labels, bronze.machine_metadata, bronze.maintenance_logs
## Data Source: NASA CMAPSS FD001 + FD002, Azure PdM dataset


In [0]:
BASE_PATH = "/Volumes/workspace/predictive_maintenance/raw_data"

# Verify files
files = dbutils.fs.ls(BASE_PATH)
for f in files:
    print(f.name, "—", round(f.size/1024/1024, 2), "MB")

In [0]:
# NASA CMAPSS columns — no header in file, we define manually
sensor_cols = [
    "unit_id", "cycle",
    "setting_1", "setting_2", "setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
    "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
    "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
    "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20",
    "sensor_21"
]
print(f"Defined {len(sensor_cols)} columns")

In [0]:
from pyspark.sql.functions import lit

# Load FD001
fd001_train = spark.read.csv(f"{BASE_PATH}/train_FD001.txt", sep=" ", inferSchema=True) \
    .drop("_c26", "_c27") \
    .toDF(*sensor_cols) \
    .withColumn("source_dataset", lit("FD001"))

fd001_test = spark.read.csv(f"{BASE_PATH}/test_FD001.txt", sep=" ", inferSchema=True) \
    .drop("_c26", "_c27") \
    .toDF(*sensor_cols) \
    .withColumn("source_dataset", lit("FD001"))

# Load FD002
fd002_train = spark.read.csv(f"{BASE_PATH}/train_FD002.txt", sep=" ", inferSchema=True) \
    .drop("_c26", "_c27") \
    .toDF(*sensor_cols) \
    .withColumn("source_dataset", lit("FD002"))

fd002_test = spark.read.csv(f"{BASE_PATH}/test_FD002.txt", sep=" ", inferSchema=True) \
    .drop("_c26", "_c27") \
    .toDF(*sensor_cols) \
    .withColumn("source_dataset", lit("FD002"))

print(f" FD001 train: {fd001_train.count()} rows")
print(f" FD001 test:  {fd001_test.count()} rows")
print(f" FD002 train: {fd002_train.count()} rows")
print(f" FD002 test:  {fd002_test.count()} rows")

In [0]:
# Union all 4 into one
all_sensors = fd001_train.union(fd001_test).union(fd002_train).union(fd002_test)

print(f" Total rows combined: {all_sensors.count()}")

# Write as Bronze Delta table
all_sensors.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.predictive_maintenance.bronze_nasa_sensor_raw")

print(" bronze_nasa_sensor_raw created!")

In [0]:
df = spark.table("workspace.predictive_maintenance.bronze_nasa_sensor_raw")
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
display(df.limit(5))

In [0]:
# Maintenance logs
maint_df = spark.read.csv(
    f"{BASE_PATH}/PdM_maint.csv",
    header=True,
    inferSchema=True
)
maint_df.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.predictive_maintenance.bronze_maintenance_logs")
print(f"bronze_maintenance_logs: {maint_df.count()} rows")

# Telemetry / machine metadata
telemetry_df = spark.read.csv(
    f"{BASE_PATH}/PdM_telemetry.csv",
    header=True,
    inferSchema=True
)
telemetry_df.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.predictive_maintenance.bronze_machine_metadata")
print(f"bronze_machine_metadata: {telemetry_df.count()} rows")

In [0]:
tables = spark.sql("SHOW TABLES IN workspace.predictive_maintenance")
display(tables)

In [0]:
tables_to_enforce = [
    "bronze_nasa_sensor_raw",
    "bronze_maintenance_logs", 
    "bronze_machine_metadata"
]

for table in tables_to_enforce:
    spark.sql(f"""
        ALTER TABLE workspace.predictive_maintenance.{table}
        SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
    """)
    print(f"Schema enforcement set on {table}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Try writing bad data — this SHOULD fail
try:
    bad_data = spark.createDataFrame(
        [("wrong", "data", "here")],
        ["col_a", "col_b", "col_c"]
    )
    bad_data.write.format("delta").mode("append") \
        .saveAsTable("workspace.predictive_maintenance.bronze_nasa_sensor_raw")
    print("Schema enforcement NOT working — bad write succeeded")
except Exception as e:
    print("Schema enforcement WORKING — bad write correctly rejected")
    print(f"   Error caught: {str(e)[:120]}")

In [0]:
# Optimize for fast queries by unit_id and cycle
spark.sql("""
    OPTIMIZE workspace.predictive_maintenance.bronze_nasa_sensor_raw
    ZORDER BY (unit_id, cycle)
""")
print("bronze_nasa_sensor_raw optimized")

spark.sql("""
    OPTIMIZE workspace.predictive_maintenance.bronze_maintenance_logs
    ZORDER BY (machineID)
""")
print("bronze_maintenance_logs optimized")

spark.sql("""
    OPTIMIZE workspace.predictive_maintenance.bronze_machine_metadata
    ZORDER BY (machineID)
""")
print("bronze_machine_metadata optimized")

In [0]:
# Check version history — this is Delta Lake magic
print("=== VERSION HISTORY ===")
spark.sql("""
    DESCRIBE HISTORY 
    workspace.predictive_maintenance.bronze_nasa_sensor_raw
""").select("version", "timestamp", "operation", "operationParameters") \
  .show(10, truncate=False)

In [0]:
# Query data AS IT WAS at version 0 — before any changes
print("=== TIME TRAVEL QUERY — Version 0 ===")
df_v0 = spark.sql("""
    SELECT * FROM workspace.predictive_maintenance.bronze_nasa_sensor_raw
    VERSION AS OF 0
    LIMIT 10
""")
display(df_v0)
print("Time travel working — this is your video demo moment!")

In [0]:
print("=" * 50)
print("DAY 1 COMPLETE — BRONZE LAYER SUMMARY")
print("=" * 50)

tables = [
    "bronze_nasa_sensor_raw",
    "bronze_maintenance_logs",
    "bronze_machine_metadata"
]

for t in tables:
    count = spark.table(f"workspace.predictive_maintenance.{t}").count()
    cols  = len(spark.table(f"workspace.predictive_maintenance.{t}").columns)
    print(f" {t}: {count:,} rows | {cols} columns")

print("=" * 50)
print("Schema enforcement: TESTED")
print("OPTIMIZE + ZORDER:  DONE")
print("Time travel:        WORKING")
print("=" * 50)